# 01 - Prepare the Darija data

This notebook prepares the corpus for our project. We use only **datasets published by AtlasIA**; we will build our own tokenizer in notebook 02.

Pipeline: choose CSV files → inspect the `darija` column → normalize and deduplicate → sample → split into training and validation → save the splits.

**The notebook is delivered unexecuted.** Downloads are disabled by default. No tokenizer or model is trained here.

## 1. Source and first corpus

Dataset: [atlasia/darija_english](https://huggingface.co/datasets/atlasia/darija_english). To start, we use `atlasia_stories.csv` and keep its `darija` column. The `english` translations are not used for training.

The dataset page lists the **CC BY-NC 4.0** license. Downloaded files stay local and are excluded from Git. We pin a dataset revision to identify the requested data.

`MAX_DOCUMENTS` sets the size of this workshop; it is a teaching parameter, not a guaranteed sufficient amount of data for a good-quality LLM.

In [ ]:
import csv
import hashlib
import json
import random
import unicodedata
from pathlib import Path
from urllib.request import urlopen

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / 'README.md').is_file() and (p / 'notebooks').is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError('Open this notebook from the darija-llm-lab project directory.')

DATASET = 'atlasia/darija_english'
REVISION = 'c6468fe948155c935c7a8ce59db7df895ad1296a'
SOURCE_FILES = ['atlasia_stories.csv']
MAX_DOCUMENTS = 1000
SEED = 42
VALIDATION_FRACTION = 0.10
DOWNLOAD_DATA = False
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
print('Project:', ROOT)
print('Expected CSV files in:', RAW_DIR)

## 2. Put the CSV files in the project

You can copy files you have already downloaded into `data/raw/`, or set `DOWNLOAD_DATA` to `True` and run the next cell. Existing files are reused and are not overwritten.

For a manually copied CSV, we record its SHA-256 hash to identify the exact file. The revision in the URL alone does not prove that a local file came from that revision.

In [ ]:
def source_url(filename):
    return (f'https://huggingface.co/datasets/{DATASET}/'
            f'resolve/{REVISION}/{filename}')

if DOWNLOAD_DATA:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for filename in SOURCE_FILES:
        destination = RAW_DIR / filename
        if destination.exists():
            print('Reusing existing file:', filename)
            continue
        with urlopen(source_url(filename), timeout=60) as response:
            payload = response.read()
        destination.write_bytes(payload)
        print('Downloaded:', filename, '-', len(payload), 'bytes')
else:
    print('Download is disabled. Use your local CSV files in data/raw/.')

## 3. Inspect the columns before processing the text

A CSV can contain several kinds of information: an ID, a translation, a source, and Darija text. We inspect its schema instead of assuming every column should be used for training.

We keep the corpus focused on Darija. Adding English translations would change the training objective and should be an explicit choice.

In [ ]:
csv_paths = [RAW_DIR / name for name in SOURCE_FILES]
missing = [str(path) for path in csv_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'CSV manquants. Copie-les dans data/raw/ ou active DOWNLOAD_DATA : '
        + ', '.join(missing)
    )

for path in csv_paths:
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        if 'darija' not in (reader.fieldnames or []):
            raise ValueError(f'Missing darija column in {path.name}.')
        print(path.name, '→', reader.fieldnames)
        first_row = next(reader, None)
        if first_row:
            print('First sample:', (first_row['darija'] or '')[:200])

## 4. Normalize without changing the language

We apply **NFC** and strip whitespace from the edges of each document. NFC brings some equivalent Unicode representations into the same form. It does not transliterate Arabizi into Arabic or correct spelling variants.

Teaching examples: `سلام، لاباس؟`, `salam, labas?`, `ghadi nji demain`. Digits such as `3` and `7`, accents, and punctuation are preserved.

We remove only exact duplicates after normalization. Paraphrases, near-duplicates, and related documents would need further analysis. The `darija` column may also contain noise or other languages; its name is not a quality guarantee.

In [ ]:
def normalize_document(text):
    return unicodedata.normalize('NFC', text).strip()

def text_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def read_unique_documents(paths):
    documents, seen, sources = [], set(), []
    for path in paths:
        counts = {'rows': 0, 'empty_rows': 0, 'duplicate_rows': 0}
        with path.open(encoding='utf-8-sig', newline='') as stream:
            reader = csv.DictReader(stream)
            if 'darija' not in (reader.fieldnames or []):
                raise ValueError(f'Missing darija column in {path.name}.')
            for row in reader:
                counts['rows'] += 1
                text = normalize_document(row.get('darija') or '')
                if not text:
                    counts['empty_rows'] += 1
                elif text in seen:
                    counts['duplicate_rows'] += 1
                else:
                    seen.add(text)
                    documents.append(text)
        sources.append({
            'filename': path.name, 'url': source_url(path.name),
            'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
            'bytes': path.stat().st_size, **counts,
        })
    return documents, sources

In [ ]:
if MAX_DOCUMENTS < 2 or not 0 < VALIDATION_FRACTION < 1:
    raise ValueError('Choose at least two documents and a fraction between 0 and 1.')

all_documents, source_manifest = read_unique_documents(csv_paths)
random.Random(SEED).shuffle(all_documents)
documents = all_documents[:MAX_DOCUMENTS]
if len(documents) < 2:
    raise ValueError('At least two distinct documents are required.')

n_valid = max(1, min(len(documents) - 1, round(len(documents) * VALIDATION_FRACTION)))
valid_texts = documents[:n_valid]
train_texts = documents[n_valid:]
assert set(train_texts).isdisjoint(valid_texts)

print('Distinct documents available:', len(all_documents))
print('Selected documents:', len(documents))
print('Training:', len(train_texts), '| Validation:', len(valid_texts))
for text in train_texts[:3]:
    print(repr(text[:160]))

## 5. Why split before tokenization?

The tokenizer learns its vocabulary and merges from corpus statistics, so validation must be held out **before** that learning step. We create sliding windows separately within each split in notebook 03.

These splits are made for our workshop; they are not an official AtlasIA evaluation split. They help us follow a sound protocol but do not guarantee that near-duplicates are absent.

In [ ]:
def write_jsonl(path, texts):
    with path.open('w', encoding='utf-8', newline='\n') as stream:
        for text in texts:
            row = {'text': text, 'sha256': text_hash(text)}
            stream.write(json.dumps(row, ensure_ascii=False) + '\n')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(PROCESSED_DIR / 'train.jsonl', train_texts)
write_jsonl(PROCESSED_DIR / 'validation.jsonl', valid_texts)
corpus_manifest = {
    'dataset': DATASET, 'requested_revision': REVISION,
    'column': 'darija', 'dataset_license': 'CC-BY-NC-4.0',
    'sources': source_manifest, 'normalization': 'NFC + strip document edges',
    'deduplication': 'exact normalized documents across selected CSV files',
    'sampling': 'shuffle unique documents, then take the configured maximum',
    'seed': SEED, 'max_documents': MAX_DOCUMENTS,
    'available_unique_documents': len(all_documents),
    'train_documents': len(train_texts), 'validation_documents': len(valid_texts),
    'validation_used_for_tokenizer_training': False,
}
(PROCESSED_DIR / 'corpus_manifest.json').write_text(
    json.dumps(corpus_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Splits saved to:', PROCESSED_DIR)